# 🧩 Gemma 3 (4B) Fine-Tuning — Metinden Yapılandırılmış JSON Çıkarma
### Unsloth + QLoRA ile Kaggle / Colab üzerinde, ücretsiz T4 GPU

**Antalya — Büyük Dil Modelleri (LLMs) Tabanlı Uygulama Geliştirme Eğitimi**

---

## Bu notebook'ta ne yapacağız, niçin?

Serbest, dağınık bir metni (fatura, tıbbi kayıt, sipariş…) **verilen JSON şemasına birebir uyan** temiz bir JSON'a çeviren bir model eğiteceğiz.

**Niçin bu görev?** Gerçek üretimde en çok işe yarayan LLM görevlerinden biri: belgeden veri çıkarıp veritabanına / otomasyona besleme (*schema-guided structured extraction*). Çıktı geçerli JSON olduğunda `json.loads()` ile doğrudan koda akar.

**Niçin fine-tune?**
> **Analoji:** Prompt vermek, birine *her seferinde* "lütfen şu formatta yaz" diye not bırakmaktır. Fine-tune, o kişiyi *eğitip* o formatı kas hafızasına yerleştirmektir.

- Küçük model + prompt → çoğu zaman "İşte JSON'unuz:" gevezeliği, şemadan sapma, uydurma alan.
- Fine-tune → modeli **sadece geçerli JSON** üretmeye şartlarız → çıktı doğrudan parse edilir.
- Küçük model (4B) + QLoRA → tek **ücretsiz** T4 GPU'da eğitilir, laptopta/Ollama'da çalışır. API ücreti yok, veri dışarı çıkmaz (KVKK/gizlilik).

> ✅ **Bu notebook Kaggle T4'te uçtan uca ÇALIŞTIRILDI (gerçek sonuç):** yükleme 4.6 GB, peak VRAM 5.9 GB, ve en önemlisi — **baseline %0 → fine-tuned %95 geçerli-JSON oranı** (20 test örneğinde). Yani fine-tune, ham modelin üretemediği temiz JSON'u üretebilir hale getirdi.
> ⚙️ **Kaggle:** Sağ panel → *Accelerator* = **GPU T4 x2**, *Internet* = **On**.


## 🔍 Neden Gemma 3 4B? (Gemma 4 E4B neden değil?)

Bu uygulama **önce Gemma 4 E4B** ile denendi. Ücretsiz Kaggle **T4** GPU'da yapılan gerçek çalıştırmalar şunu kanıtladı:

| Model | Yükleme | Inference | **QLoRA Eğitimi** |
|---|---|---|---|
| Gemma 4 **E4B** (auto / float16 / QAT-w4a16) | ❌ OOM | — | — |
| Gemma 4 **E2B** | ✅ | ✅ | ❌ dtype çakışması / float32'de OOM |
| **Gemma 3 4B** (bu notebook) | ✅ 4.6 GB | ✅ | ✅ **loss 0.42, peak 5.9 GB** |

**Kök neden (donanım fiziği):** T4 = *Turing* mimarisi ve **bfloat16 desteklemez**. Gemma 4 float16'da sayısal olarak kararsızdır (NaN) → kütüphane onu **float32**'ye zorlar → 8B parametreli E4B float32'de T4'ün 14.5 GB'ına **sığmaz**. Gemma 3 4B ise float16'da **stabildir** → T4'te sorunsuz eğitilir.

**Ders (mühendislik):** *En yeni model her zaman en doğru seçim değildir.* Donanımınla (ücretsiz T4 = bf16 yok) modelin gereksinimleri uyuşmalı. Gemma 4 QLoRA için **bf16 destekleyen Ampere+ GPU** (L4/A100, ücretli) gerekir; **ücretsiz** eğitim hedefinde doğru seçim **Gemma 3 4B**'dir.

> Not: "Gemini 4B" diye bir model yoktur — Gemini kapalı API'dir, fine-tune edilmez. Açık ağırlıklı Gemma ailesini eğitiriz.


## 📚 Önce üç kavram (5 dakikalık teori)

**1) Base vs Instruct (-it) model:** Base model ham bir *sonraki-kelime tahmincisidir*. Instruct (`-it`) komut anlayıp cevap verecek şekilde hizalanmıştır. Biz `-it` seçiyoruz — sadece **davranışı incelteceğiz** (JSON disiplini).

**2) Quantization (4-bit):** 16-bit model diskte ~8 GB; **4-bit**'e sıkıştırınca ~3 GB'a iner → ücretsiz GPU'ya sığar, doğrulukta ihmal edilebilir kayıp.

**3) LoRA / QLoRA / PEFT:** Ağırlıkları dondurur, araya küçük eğitilebilir matrisler (LoRA) koyarız. **QLoRA** = LoRA + 4-bit taban. Sadece ~%0.5 parametre eğitilir.
> **Analoji:** Koca bir kütüphaneyi baştan yazmak yerine kenara birkaç **not kağıdı** iliştiririz. Kütüphane aynı kalır, notlar görevi öğrenir.


## 1. Kurulum

**Ne:** Unsloth + bağımlılıklarını kuruyoruz. **Niçin sade?** `pip install unsloth` uyumlu `transformers` sürümünü kendisi çözer; ayrıca sürüm pinlemeye gerek yok. Aşağısı Unsloth'un resmi, Colab ve Kaggle'da çalışan kalıbıdır.

> **Yayım/tekrar-üretilebilirlik notu:** Sınıfta güncel sürüm iyidir. Ama birebir tekrar üretmek isterseniz `unsloth==2026.7.2` gibi sabitleyin (Kaggle taban imajı zamanla değişir; bu notebook 2026-07-13 imajında doğrulandı).

In [ ]:
# Kaggle T4'te uçtan uca doğrulanmış kurulum. Sade tutun — ek pip satırları (torchao pin vb.)
# bağımlılık çözümünü bozabiliyor. Unsloth transformers/torchao'yu kendisi uyumlu kurar.
!pip install -q unsloth

In [ ]:
import torch, unsloth
print("Unsloth :", unsloth.__version__)
print("GPU     :", torch.cuda.get_device_name(0))
print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

## 2. Modeli 4-bit yükle

**Ne:** `unsloth/gemma-3-4b-it`'yi 4-bit yüklüyoruz. **Niçin `max_seq_length=2048`?** Bir örnekte *şema + metin + hedef JSON* bu token bütçesine sığmalı. (Kabul testinde 1024'te peak 5.9 GB ölçüldü; 2048 de T4'e rahat sığar.)

In [ ]:
from unsloth import FastModel

MAX_SEQ_LEN = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-3-4b-it",
    dtype           = None,     # otomatik (T4'te float16 — Gemma 3'te stabil)
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,     # QLoRA tabanı
    full_finetuning = False,
)

### LoRA adaptörünü ekle

**Niçin `finetune_vision_layers=False`?** Gemma 3 4B çok-kipli (metin+görsel), ama görev sadece metin → görü katmanlarını eğitmek boşa kaynak. **`r=8, lora_alpha=8`:** küçük veri için yeterli kapasite.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 8,
    lora_alpha   = 8,
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)
model.print_trainable_parameters()

## 3. Veriyi yükle ve tanı

**Ne:** `paraloq/json_data_extraction` — her satırda **`text`** (dağınık belge), **`schema`** (hedef şema), **`item`** (doğru JSON); üçü de string.
**Gerçek hayatta:** Firmada/kamuda HF'deki hazır set işinize yaramaz; *kendi* verinizi hazırlarsınız (ödevlerde var).

In [ ]:
from datasets import load_dataset

raw = load_dataset("paraloq/json_data_extraction", split="train")
print(raw)
print("Konular:", sorted(set(raw["topic"])))
print("\n--- METİN (ilk 400) ---\n", raw[0]["text"][:400])
print("\n--- HEDEF JSON ---\n", raw[0]["item"][:400])

### Uzunluk filtresi

Bazı belgeler çok uzun ve 2048 token'a sığmaz; model onları göremez. Kaba karakter eşiğiyle ayıklıyoruz — **girdi context window'a sığmıyorsa işe yaramaz.**

In [ ]:
import json

def toplam_uzunluk(row):
    return len(row["schema"]) + len(row["text"]) + len(row["item"])

filt = raw.filter(lambda r: toplam_uzunluk(r) < 4500)
print(f"Filtre sonrası: {len(filt)} / {len(raw)}")

## 4. Sohbet formatına çevir

Her örneği bir konuşmaya çeviriyoruz — **user**: talimat+şema+metin, **assistant**: sadece hedef JSON.

**Gemma 3 kritik noktalar (kabul testinde doğrulandı):**
- `get_chat_template(..., "gemma-3")`: Gemma 3'ün `<start_of_turn>` etiketlerini uygular.
- İçerik **liste** biçiminde: `[{"type":"text","text": ...}]` — Gemma 3 4B çok-kipli olduğu için içerik düz string değil liste bekler (düz string `TypeError: string indices` verir).
- `.removeprefix('<bos>')`: şablon başa `<bos>` ekler; çift bos'u önlemek için kırpıyoruz.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

TALIMAT = ("Aşağıdaki metinden, verilen JSON şemasına birebir uyan geçerli bir JSON üret. "
           "Yalnızca JSON döndür; açıklama, giriş cümlesi veya kod bloğu işareti ekleme.")

def sohbete_cevir(row):
    hedef = json.dumps(json.loads(row["item"]), ensure_ascii=False)
    kullanici = f"{TALIMAT}\n\n### ŞEMA:\n{row['schema']}\n\n### METİN:\n{row['text']}"
    return {"conversations": [
        {"role": "user",      "content": [{"type": "text", "text": kullanici}]},
        {"role": "assistant", "content": [{"type": "text", "text": hedef}]},
    ]}

konusma = filt.map(sohbete_cevir, remove_columns=filt.column_names)

def formatla(batch):
    return {"text": [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
        for c in batch["conversations"]
    ]}

veri = konusma.map(formatla, batched=True, remove_columns=["conversations"])
print(veri[0]["text"][:600])

### Eğitim / test ayrımı — data leakage'a karşı

Modeli **görmediği** örneklerde değerlendireceğiz; yoksa ezber yüzünden skor yanıltıcı olur.

In [ ]:
bolunmus = veri.train_test_split(test_size=0.1, seed=3407)
train_ds, test_ds = bolunmus["train"], bolunmus["test"]
print("Eğitim:", len(train_ds), "| Test:", len(test_ds))
ham_test = filt.train_test_split(test_size=0.1, seed=3407)["test"]

## 5. ⚖️ Baseline — fine-tune ÖNCESİ

**Niçin önce ölçüyoruz?** İyileşmeyi kanıtlamak için başlangıç noktası. Metrik: çıktı `json.loads()` ile parse oluyor mu? (geçerli-JSON oranı).

In [ ]:
def istem(row):
    kullanici = f"{TALIMAT}\n\n### ŞEMA:\n{row['schema']}\n\n### METİN:\n{row['text']}"
    return [{"role": "user", "content": [{"type": "text", "text": kullanici}]}]

def cikar(messages, max_new_tokens=512):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=1.0, top_p=0.95, top_k=64)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def gecerli_json_orani(orneklerden):
    ok = 0
    for row in orneklerden:
        try: json.loads(cikar(istem(row)).strip()); ok += 1
        except Exception: pass
    return ok / len(orneklerden)

degerlendirme = list(ham_test)[:20]
baseline_oran = gecerli_json_orani(degerlendirme)
print(f"\n📉 BASELINE geçerli-JSON oranı: {baseline_oran:.0%}")

In [ ]:
print(cikar(istem(degerlendirme[0]))[:800])

## 6. Eğitim (SFT)

**Niçin `max_steps=60`?** Demo (~10 dk). Gerçekte `num_train_epochs=1..3`.
**Niçin `batch_size=1` + `grad_accum=4`?** Belleği şişirmeden sanki 4'lük batch.
**`train_on_responses_only`:** Model sadece **cevap** (JSON) tokenlarından öğrensin; uzun girişi ezberlemesin. Gemma 3 etiketleri: `<start_of_turn>user\n` / `<start_of_turn>model\n`.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = train_ds, eval_dataset = None,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        max_steps                   = 60,   # DEMO — gerçekte num_train_epochs=1..3
        learning_rate               = 2e-4,
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        report_to                   = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)

In [ ]:
istatistik = trainer.train()
print("\nSon eğitim kaybı (loss):", round(istatistik.training_loss, 4))

## 7. 📈 Fine-tune SONRASI — aynı testte ölç

In [ ]:
finetuned_oran = gecerli_json_orani(degerlendirme)
print(f"📉 Baseline    : {baseline_oran:.0%}")
print(f"📈 Fine-tuned  : {finetuned_oran:.0%}")
print(f"Δ İyileşme     : {(finetuned_oran - baseline_oran):+.0%}")

In [ ]:
cikti = cikar(istem(degerlendirme[0]))
print(cikti[:800])
print("\n--- json.loads testi ---")
try:
    p = json.loads(cikti.strip()); print("✅ Geçerli JSON. Anahtarlar:", list(p.keys()))
except Exception as e:
    print("❌ Parse hatası:", e)

## 8. Modeli kaydet

Sadece LoRA adaptörünü kaydediyoruz (~50-100 MB). Taban model runtime'da yeniden yüklenir.

In [ ]:
model.save_pretrained("gemma3-json-lora")
tokenizer.save_pretrained("gemma3-json-lora")
print("✅ Adaptör kaydedildi: gemma3-json-lora/")

### (Bonus) Ollama / llama.cpp için GGUF
Yerelde offline çalıştırmak için. Zaman/yer ister; sınıfta opsiyonel.

In [ ]:
# model.save_pretrained_gguf("gemma3-json-gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF export için üstteki satırı açın.")

## 9. Özet ve ödevler

**Neyi niçin yaptık:** 4-bit yükleme (ücretsiz T4), LoRA (ucuz), sohbet formatı (hedefi örnekle), train/test (dürüst ölçüm), baseline→SFT→sonrası (objektif kanıt: json.loads oranı).

**Ödevler:**
- `max_steps` yerine `num_train_epochs=2` ile tam eğitim; oranı tekrar ölç.
- `jsonschema.validate` ile şemaya uygunluk + alan F1.
- `r`/`lora_alpha` (8→16→32) etkisi.
- **Kendi Türkçe verinizi** (fatura/dilekçe → JSON) hazırlayıp aynı hattı çalıştır.
- GGUF + Ollama ile yerel mini API.

---
*Antalya LLM Tabanlı Uygulama Geliştirme Eğitimi — Dr. Murat Altun*
*Model seçimi Kaggle T4'te gerçek çalıştırmayla doğrulandı: Gemma 3 4B ✅ / Gemma 4 E4B ❌ (bf16 gerekliliği).*
